# OffScript — Phase 1: Exploratory Analysis

This notebook explores pitch selection patterns across the full 15-pitcher roster
using the combined dataset produced in notebook 02. Analysis is driven by utility
functions from `src/pitch_analysis.py`.

**Goals:**
- Establish a league-average pitch usage baseline by count state
- Examine each pitcher's individual usage profile against that baseline
- Identify early deviation candidates through side-by-side comparisons
- Produce pitch location charts for all 15 pitchers

**Input:** `data/processed/pitcher_data.parquet`  
**Output:** Figures saved to `reports/figures/`

## 1. Setup

In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
import matplotlib.pyplot as plt
from pitch_analysis import (
    load_data,
    get_pitcher,
    pitch_usage_by_count,
    pitch_usage_by_handedness,
    plot_pitch_locations,
    plot_usage_comparison,
)

data = load_data()

print(f"Dataset loaded: {len(data):,} pitches, {data['pitcher_name'].nunique()} pitchers")

## 2. League-Average Pitch Usage by Count

Computing usage across all 15 pitchers combined establishes a baseline that
reflects conventional pitch selection patterns. This baseline is the foundation
the Phase 2 model is trained to approximate — deviations from it are what
Phase 3 quantifies.

In [ ]:
league_avg = pitch_usage_by_count(data)

print("League-average pitch usage by count state:")
print(
    league_avg.pivot(index="count", columns="pitch_type", values="pct")
    .fillna(0)
    .round(1)
)

## 3. Individual Pitcher Usage Profiles

Each pitcher's usage table is printed for visual inspection. Pitchers whose
profiles diverge noticeably from the league average — particularly in high-leverage
counts like 0-2 and 3-0 — are strong candidates for the deviation analysis in
Phase 3.

In [ ]:
for name in data["pitcher_name"].unique():
    pitcher = get_pitcher(data, name)
    usage = pitch_usage_by_count(pitcher)
    print(f"\n{'=' * 40}")
    print(f"  {name}")
    print(f"{'=' * 40}")
    print(
        usage.pivot(index="count", columns="pitch_type", values="pct")
        .fillna(0)
        .round(1)
    )

## 4. Side-by-Side Pitcher Comparisons

Cole versus Cease is the most instructive pairing in the roster: Cole represents
conventional multi-pitch sequencing while Cease is the clearest deviation
candidate due to his extreme slider reliance across all count states.

In [ ]:
plot_usage_comparison(data, "Gerrit Cole", "Dylan Cease")

## 5. Pitch Location Analysis

Location charts are generated for all 15 pitchers and saved to `reports/figures/`.
These charts reveal whether a pitcher's pitch type distribution reflects deliberate
location tendencies — a complementary lens to the usage analysis above.

In [ ]:
for name in data["pitcher_name"].unique():
    pitcher = get_pitcher(data, name)
    plot_pitch_locations(pitcher, name)